# 01 — Data Exploration

Explore the `ccdv/arxiv-summarization` dataset and understand:
- Distribution of paper categories
- Article and abstract length distributions
- Sample papers from AI Engineering categories

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt

# Load a small sample for exploration (streaming avoids full download)
ds = load_dataset('ccdv/arxiv-summarization', split='train', streaming=True, trust_remote_code=True)
sample = [next(iter(ds)) for _ in range(500)]  # grab 500 examples
df = pd.DataFrame(sample)
print(df.columns.tolist())
df.head(3)

In [ ]:
# Article and abstract length distributions
df['article_len'] = df['article'].str.split().str.len()
df['abstract_len'] = df['abstract'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['article_len'].clip(upper=5000).hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Article Length (words, capped at 5000)')
axes[0].set_xlabel('Words')

df['abstract_len'].clip(upper=400).hist(bins=40, ax=axes[1], color='darkorange')
axes[1].set_title('Abstract Length (words)')
axes[1].set_xlabel('Words')
plt.tight_layout()
plt.show()

print(f"Median article length : {df['article_len'].median():.0f} words")
print(f"Median abstract length: {df['abstract_len'].median():.0f} words")

In [ ]:
# Category distribution
AI_CATS = {'cs.LG', 'cs.AI', 'cs.CL', 'cs.CV', 'stat.ML'}

df['is_ai'] = df['categories'].apply(
    lambda cats: bool(set(str(cats).split()) & AI_CATS) if cats else False
)
print(f"AI Engineering papers: {df['is_ai'].sum()} / {len(df)} ({df['is_ai'].mean()*100:.1f}%)")

# Top categories
all_cats = [cat for cats in df['categories'].dropna() for cat in str(cats).split()]
cat_counts = pd.Series(all_cats).value_counts().head(20)
cat_counts.plot(kind='barh', figsize=(10, 6), color='steelblue')
plt.title('Top 20 arXiv Categories in Sample')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Sample 3 AI Engineering papers
ai_papers = df[df['is_ai']].head(3)
for _, row in ai_papers.iterrows():
    print(f"Categories: {row['categories']}")
    print(f"Abstract  : {row['abstract'][:300]}...")
    print(f"Article snippet: {row['article'][:200]}...")
    print('-' * 80)